In [ ]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2
import torch, rdkit
import os
from pathlib import Path
import sys
def _is_models_root(p: Path) -> bool:
    return (p / "utils").is_dir() and (p / "models").is_dir() and (p / "notebooks").is_dir() and (p / "data").is_dir()


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if _is_models_root(p):
            return p
        cand = p / "MY_PAPER_RELATED" / "MODELS"
        if _is_models_root(cand):
            return cand
    raise FileNotFoundError("Could not locate MODELS project root")
PROJECT_ROOT = resolve_project_root()
os.environ["MODELS_VARIANT"] = "Encoder_Only_PSMILES"
os.environ["MODELS_FULL_DATA_TRAINING"] = "1"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rdkit import Chem
from utils.utils import *
from utils.dataloader import *
from utils.loss_fn import *
from utils.eval import *

from tqdm import trange
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sample_ps = PS("CC(CCCNC(=O)C(C)OC(=O)[*])O[*]").canonicalize.psmiles
if getattr(dataset, "token_mode", "selfies") == "psmiles":
    print(sample_ps)
    print(split_psmiles_tokens(sample_ps))
else:
    sample_sf = sfp.encoder_psmiles(sample_ps, strict=False)
    print(sample_sf)
    print(list(sfp.split_selfies(sample_sf)))
vocab = dataset.vocab
index_to_token = {idx: token for token, idx in vocab.items()}
print("Split summary:")
print(split_summary.to_string(index=False))
print("Split leakage counts:", split_info["leak_counts"])
print(f"Split tag: {SPLIT_TAG}")
print(f"Full-data training mode: {FULL_DATA_TRAINING}")
print(f"Dataset sizes - train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")


In [ ]:
from models.Trans import Encoder_Only

mode = "Encoder_Only_PSMILES"
model_cls = Encoder_Only
model = model_cls().to(device)

PAD_IDX = dataset.vocab['[PAD]']
loss_fn = nn.CrossEntropyLoss(reduction="mean", ignore_index=PAD_IDX)


In [ ]:
lr = 3e-4
from torch.optim import AdamW
optim = AdamW(model.parameters(), lr=lr)


In [ ]:
import datetime
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.error')

epoch = 2000
VAL_EVERY = 10
model.train()
progress = tqdm(range(epoch), desc="Training")
loss_arr = []
val_loss_arr = []

for i in progress:
    batchloss = 0.0
    model.train()
    for batch in train_dataloader:
      sm_enc, sm_dec_in, sm_dec_out, cond_scalar = [t.to(device) for t in batch]
      optim.zero_grad()

      attn_mask = sm_dec_in == PAD_IDX
      output = model(sm_dec_in, cond_scalar, attn_mask=attn_mask)
      loss = loss_fn(output.reshape(-1, dataset.vocab_size), sm_dec_out.view(-1))
      loss.backward()
      optim.step()
      batchloss += loss.item()

    train_loss = batchloss / max(1, len(train_dataloader))
    loss_arr.append(train_loss)

    val_loss = val_loss_arr[-1] if val_loss_arr else float("nan")
    if (i + 1) % VAL_EVERY == 0 or i == epoch - 1:
        val_batchloss = 0.0
        model.eval()
        with torch.no_grad():
            for batch in val_dataloader:
                sm_enc, sm_dec_in, sm_dec_out, cond_scalar = [t.to(device) for t in batch]
                attn_mask = sm_dec_in == PAD_IDX
                output = model(sm_dec_in, cond_scalar, attn_mask=attn_mask)
                val_batchloss += loss_fn(
                    output.reshape(-1, dataset.vocab_size),
                    sm_dec_out.view(-1),
                ).item()
        val_loss = val_batchloss / max(1, len(val_dataloader))
        val_loss_arr.append(val_loss)
        model.train()

    progress.set_description(f"train_loss={train_loss:.6f} val_loss={val_loss:.6f}")


In [ ]:
model.eval()
results = []
origin = []

EVAL_SPLIT_NAME = "test"
EVAL_DATASET = test_dataset
EVAL_DATALOADER = test_dataloader

print(EVAL_SPLIT_NAME, len(EVAL_DATASET))
with torch.no_grad():
    for (smiles_enc, smiles_dec_input, smiles_dec_output, cond_scalar) in EVAL_DATALOADER:
        smiles_enc = smiles_enc.to(device)
        smiles_dec_input = smiles_dec_input.to(device)
        smiles_dec_output = smiles_dec_output.to(device)
        cond_scalar = cond_scalar.to(device)

        attn_mask = smiles_dec_input == PAD_IDX
        result = model(smiles_dec_input, cond_scalar, attn_mask=attn_mask)

        results.append(result)
        origin.append(smiles_dec_output)

results = torch.cat(results, dim=0)
origin = torch.cat(origin, dim=0)
results = nn.functional.softmax(results, dim=-1)
argmax_indices = torch.argmax(results, dim=-1)
output = torch.nn.functional.one_hot(argmax_indices, num_classes=results.size(-1))

from sklearn.metrics import mean_absolute_error

results_smiles = []
origin_smiles = []

for row in argmax_indices:
    smiles = tok_ids_to_smiles(row.tolist(), index_to_token)
    results_smiles.append(smiles or "")

for row in origin:
    smiles = tok_ids_to_smiles(row.tolist(), index_to_token)
    origin_smiles.append(smiles or "")


In [ ]:
origin_smiles = [smiles.removesuffix("EOS").strip() for smiles in origin_smiles]
results_smiles = [smiles.removesuffix("EOS").strip() for smiles in results_smiles]

for i in range(len(results_smiles)):
    if(origin_smiles[i] != results_smiles[i]):
        print(i, "번째 다름!")
    print("real smiles      : ", origin_smiles[i])
    print("predicted smiles : ", results_smiles[i])


MAE = mean_absolute_error(origin.cpu(), torch.argmax(results.cpu(), dim=-1))
print("MAE : ", MAE)



In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import DataStructs, rdFingerprintGenerator
RDLogger.DisableLog('rdApp.error')

generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def tanimoto_similarity(smiles1, smiles2):
    mol1 = Chem.MolFromSmiles(smiles1)
    mol2 = Chem.MolFromSmiles(smiles2)
    fp1 = generator.GetFingerprint(mol1)
    fp2 = generator.GetFingerprint(mol2)
    return DataStructs.TanimotoSimilarity(fp1, fp2)

def is_valid(smiles):
    return Chem.MolFromSmiles(smiles) is not None

TS = 0.0
canbe = 0
notbe = 0

for sm, orig in zip(results_smiles, origin_smiles):
    if(len(sm)==0):
        notbe += 1
        continue
    try:
        sm = PS(sm).canonicalize.psmiles
        if is_valid(sm) and is_valid(orig):
            sim = tanimoto_similarity(sm, orig)
            TS += sim
            canbe += 1
        else:
            notbe += 1
    except:
        notbe += 1
        continue


if canbe > 0:
    print("Tanimoto Similarity : ", TS / canbe)
else:
    print("No valid molecules to compare.")

print("가능한 분자 개수 :", canbe)
print("불가능한 분자 개수 :", notbe)
print("Valid fraction      :", canbe / len(results_smiles))


In [ ]:
save_path = (PROJECT_ROOT / "checkpoints" / f"encoder_only_{mode}.pth")
torch.save(model.state_dict(), save_path)